<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/02-rdd-fundamentals/00-creating-rdds.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 2.0 — Creating RDDs: `parallelize` and `textFile`

Companion notebook to `lesson.md`. Run cells top to bottom.

Setup cell is the Module 1 boilerplate (lesson 1.2).

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("2.0-creating-rdds")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

sc = spark.sparkContext   # RDD API lives here (lesson 0.4)
sc

## `parallelize` — RDD from a Python collection

In [ ]:
rdd = sc.parallelize(range(1, 1_000_001))
print("partitions:", rdd.getNumPartitions())   # = your core count on local[*]
print("defaultParallelism:", sc.defaultParallelism)
rdd.take(5)

In [ ]:
# Control the partition count explicitly
rdd4 = sc.parallelize(["a", "b", "c", "d", "e", "f"], numSlices=4)
rdd4.getNumPartitions()

In [ ]:
# glom(): x-ray vision — which element lives in which partition?
rdd4.glom().collect()

## `textFile` — RDD from files

One string per line. Path is relative to where Jupyter was launched — adjust if needed.

In [ ]:
LOGS = f"{DATA_DIR}/sample_logs.txt"   # repo-root data/ from this notebook's folder

lines = sc.textFile(LOGS)
print("lines:", lines.count())
print("partitions:", lines.getNumPartitions())
lines.first()

In [ ]:
# Laziness preview (lesson 2.1): a bad path does NOT fail here...
ghost = sc.textFile("no/such/file.txt")
print("no error yet — the RDD is just a plan")

# ...it fails when an action forces the read. Uncomment to see the real error:
# ghost.count()

## Actions for getting data out — and their safety profile

In [ ]:
print(lines.take(3))    # safe peek: reads as little as possible
print(lines.count())    # full computation, tiny result

# collect() is fine ONLY because we know this file is 40 lines:
all_lines = lines.collect()
len(all_lines)

## Exercises

1. Create `sc.parallelize(range(100), numSlices=3)` and use `glom()` to see how Spark split the range. Are the partitions equal-sized?
2. Re-read the log file with `minPartitions=8`. Did you get exactly 8? (Hint: it's a *minimum* hint, not a command.)
3. Use `take(1)` vs `collect()` on the million-element RDD and feel the difference; check the Spark UI (`spark_ui(spark)` — it prints a link that works from Colab) — how many tasks did each launch?

In [ ]:
# your exercise code here
